# Basic Transformations

Hands-on introduction to PySpark DataFrame transformations using the employee dataset — covers `select()`, `col()`, `expr()`, and `selectExpr()` for column selection and expression-based computation.


## What is Transformation?

A **transformation** is an operation that creates a new DataFrame from an existing one — it modifies, filters, or combines data without immediately executing anything. Spark **records** the transformation as part of a logical plan and defers execution until an **action** (e.g., `show()`, `collect()`, `count()`, `write()`) triggers computation.

> [!IMPORTANT]
> Transformations are **lazy** — they build up a plan but don't run until an action is called. Actions are **eager** — they trigger execution of the entire plan.

**Common transformations:**

| Category | Examples | Description |
|----------|----------|-------------|
| Selection | `select()`, `selectExpr()`, `drop()` | Pick or remove columns |
| Filtering | `filter()`, `where()` | Keep rows matching a condition |
| Adding / Modifying | `withColumn()`, `withColumnRenamed()` | Add new columns or rename existing ones |
| Aggregation | `groupBy().agg()`, `count()`, `sum()`, `avg()` | Summarize data across groups |
| Sorting | `orderBy()`, `sort()` | Reorder rows |
| Combining | `join()`, `union()`, `unionByName()` | Merge multiple DataFrames |

**Narrow vs Wide transformations:**

| Type | Description | Examples |
|------|-------------|----------|
| **Narrow** | Data stays within a single partition — no data movement | `select()`, `filter()`, `withColumn()` |
| **Wide** | Data must move across partitions (shuffle) | `groupBy()`, `join()`, `orderBy()`, `distinct()` |

> [!TIP]
> 💡 Narrow transformations are fast and cheap — they process each partition independently. Wide transformations are more expensive because they require a **shuffle** (network data transfer between executors). Minimize wide transformations when possible.


### What are Columns?

A **Column** in PySpark is an **expression** — it represents a value (or a computation on values) for each row in a DataFrame. Columns are the building blocks of transformations: `select()`, `filter()`, `withColumn()`, and aggregations all operate on column expressions.

> [!NOTE]
> Columns are **lazy** — they describe *what* to compute, not the result. The computation only happens when an action triggers execution.

**Ways to reference a column:**

| Method | Example | Notes |
|--------|---------|-------|
| `col()` | `col("age")` | Most common — works anywhere |
| `expr()` | `expr("age + 5")` | SQL expression string |
| DataFrame subscript | `df["age"]` | Shorthand on the DataFrame |
| `F.col()` | `F.col("age")` | Using `pyspark.sql.functions` alias |
| `selectExpr()` | `df.selectExpr("age + 5 AS age_plus_5")` | SQL expressions with aliasing directly on DataFrame |

**Column expressions can include:**

- Arithmetic — `col("age") + 5`, `col("salary") * 1.10`
- Comparisons — `col("age") > 30`, `col("name").contains("Kumar")`
- String ops — `col("name").upper()`, `col("address").split(", ")`
- Conditionals — `when(col("age") > 30, "Senior").otherwise("Junior")`

> [!TIP]
> 💡 Import `col` and `expr` from `pyspark.sql.functions` — they are the most frequently used column constructors in PySpark code.
>
> ```python
> from pyspark.sql.functions import col, expr
>
> df.select(col("age") + 5)              # column expression
> df.select(expr("age + 5"))            # same result via SQL string
> df.selectExpr("age + 5 AS age_plus_5") # SQL string with alias — no expr() import needed
> ```

In [0]:
# Path to the employee CSV dataset
EMPLOYEE_DATA = "/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data/employees-data/employees.csv"

In [0]:
from pyspark.sql import SparkSession

# Create a SparkSession — the entry point for DataFrame and SQL operations
spark = SparkSession.builder.appName("basic-transformations").getOrCreate()

In [0]:
# Read the CSV with header and PERMISSIVE mode (bad rows become null instead of failing)
employee_df = (
    spark.read.format("csv")
              .option("header", True)
              .option("mode", "PERMISSIVE")
              .option("path", EMPLOYEE_DATA)
              .load()
)

employee_df.printSchema()
employee_df.show(20)


## SELECT — String Literal

Pass column names directly as strings. Simplest form — no imports needed, but no arithmetic or expressions.

In [0]:
# Select a single column by passing its name as a string literal
employee_df.select("id").show(5)

In [0]:
# Select multiple columns by passing names as string literals
employee_df.select("id", "name").show(5)


## SELECT — `col()` / `column()` Function

- Use `col()` for explicit column references — it supports arithmetic, aliases, and expressions via Python operators.
- `col()` is an alias for `column()` — both are interchangeable, but `col()` is the convention used throughout most PySpark codebases and documentation.
- Import either from `pyspark.sql.functions`:

> ```python
> from pyspark.sql.functions import col, column
>
> df.select(col("age"))      # preferred
> df.select(column("age"))  # identical result
> ```

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, column, expr, lit

# col() and column() are interchangeable — both return a Column reference
employee_df.select(col("id")).show(3)

employee_df.select(column("id")).show(3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit

# Arithmetic on a column — alias() renames the output column
employee_df.select(col("id"), (col("id") + 5).alias("id_plus_5")).show(3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit

# lit() creates a literal/constant value — useful for type-safe arithmetic
employee_df.select(col("id"), (col("id") + lit(5)).alias("id_plus_5")).show(3)


## SELECT — `expr()` Function

Use `expr()` to write SQL-style expressions as strings. Supports arithmetic, aliases (`AS`), and SQL functions inside `select()`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit

# expr() accepts SQL-style string expressions — supports AS aliasing inside the string
employee_df.select("id", expr("age + 5"), expr("age + 5 AS age_plus_5")).show(3)


## SELECT — `selectExpr()` Shorthand

Shorthand for `select(expr(...))` — accepts SQL expression strings directly on the DataFrame. No `expr()` import needed, supports `AS` aliasing.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit

# selectExpr() is shorthand for select(expr(...)) — no expr() import needed
employee_df.selectExpr("id", "age + 5", "age + 5 AS age_plus_5").show(3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit

# ❌ select("id + 5") fails with AnalysisException — string literals are column names, not expressions
# employee_df.select("id + 5").show(3)

# ✅ Resolution 1 — use col() explicitly for arithmetic
employee_df.select("id", col("id") + 5).show(3)

# ✅ Resolution 2 — use expr() for SQL-style expressions (with aliasing)
employee_df.select("id", expr("id + 5"), expr("id + 5 AS id_plus_5_1"), expr("id + 5").alias("id_plus_5_2")).show(3)

# ✅ Resolution 3 — selectExpr() shorthand for Resolution 2
employee_df.selectExpr("id", "id + 5 AS id_plus_5").show(3)



## Spark SQL

Any DataFrame operation can also be expressed as **Spark SQL** — standard SQL syntax that produces the same logical plan. To use SQL on a DataFrame:

1. **Register a temporary view** — `df.createOrReplaceTempView("view_name")` exposes the DataFrame as a SQL table
2. **Query with `spark.sql()`** — write standard `SELECT`, `WHERE`, `JOIN`, etc. against the registered view

> [!NOTE]
> Temp views are **session-scoped** — they exist only in the current Spark session and are not persisted to catalog. Use `CREATE OR REPLACE GLOBAL TEMP VIEW` for cross-session access.
>
> ```python
> employee_df.createOrReplaceTempView("employees_tbl")
>
> spark.sql("SELECT id, name, age + 5 AS age_plus_5 FROM employees_tbl").show(5)
> ```

In [0]:
# Register the DataFrame as a temp view so Spark SQL can query it
employee_df.createOrReplaceTempView("employees_tbl")

In [0]:
# Query the registered view using standard Spark SQL
dspark.sql("""
       SELECT id, name, age, age + 5 AS age_plus_5
       FROM employees_tbl;   
""").show(5)